# Course 15: ML Pipelines on Google Cloud

**GCP Professional Machine Learning Engineer — Exam Section 5**

This notebook provides hands-on practice with:
- KFP (Kubeflow Pipelines) v2 SDK — define, compile, and run pipelines
- TFX — minimal pipeline with ExampleGen + Trainer
- Vertex AI Pipelines — compile and submit (commented for cost)
- ML Metadata — query artifact lineage
- Cloud Composer — DAG definition example
- Pipeline caching and conditional execution patterns

---

## Setup & Installations

In [ ]:
# Install required packages
!pip install -q kfp==2.7.0 google-cloud-aiplatform tfx ml-metadata

In [ ]:
import warnings
warnings.filterwarnings('ignore')

print("Imports ready.")

---
## 1. Simple KFP Pipeline: Define, Compile, Run Locally

KFP v2 lets you define pipeline components as Python functions with typed inputs and outputs.
The pipeline is compiled to a JSON/YAML spec that can run on Vertex AI or any KFP-compatible backend.

In [ ]:
from kfp import dsl
from kfp.dsl import Input, Output, Dataset, Model, Metrics


# ---------- Component 1: Generate synthetic data ----------
@dsl.component(base_image="python:3.10", packages_to_install=["pandas", "scikit-learn"])
def generate_data(samples: int, output_data: Output[Dataset]):
    """Generate a synthetic binary classification dataset."""
    import pandas as pd
    from sklearn.datasets import make_classification

    X, y = make_classification(
        n_samples=samples, n_features=10,
        n_informative=5, random_state=42
    )
    df = pd.DataFrame(X, columns=[f"feat_{i}" for i in range(10)])
    df["label"] = y
    df.to_csv(output_data.path, index=False)
    print(f"Generated {len(df)} samples with {df.shape[1]-1} features.")


# ---------- Component 2: Train a model ----------
@dsl.component(base_image="python:3.10", packages_to_install=["pandas", "scikit-learn"])
def train_sklearn_model(
    training_data: Input[Dataset],
    model_artifact: Output[Model],
    metrics: Output[Metrics],
    n_estimators: int = 50,
    learning_rate: float = 0.1,
):
    """Train a GradientBoosting classifier and log metrics."""
    import pandas as pd
    import pickle
    from sklearn.ensemble import GradientBoostingClassifier
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, f1_score

    df = pd.read_csv(training_data.path)
    X = df.drop(columns=["label"])
    y = df["label"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    clf = GradientBoostingClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        random_state=42
    )
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)

    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)

    # Log metrics
    metrics.log_metric("accuracy", round(acc, 4))
    metrics.log_metric("f1_score", round(f1, 4))

    # Save model
    with open(model_artifact.path, "wb") as f:
        pickle.dump(clf, f)

    print(f"Model trained — accuracy: {acc:.4f}, F1: {f1:.4f}")


# ---------- Component 3: Evaluate and decide ----------
@dsl.component(base_image="python:3.10", packages_to_install=["pandas", "scikit-learn"])
def evaluate_model(
    model_artifact: Input[Model],
    test_data: Input[Dataset],
    threshold: float = 0.85,
) -> bool:
    """Return True if model accuracy exceeds threshold."""
    import pandas as pd
    import pickle
    from sklearn.metrics import accuracy_score

    df = pd.read_csv(test_data.path)
    X = df.drop(columns=["label"])
    y = df["label"]

    with open(model_artifact.path, "rb") as f:
        model = pickle.load(f)

    acc = accuracy_score(y, model.predict(X))
    passed = acc >= threshold
    print(f"Evaluation: accuracy={acc:.4f}, threshold={threshold}, passed={passed}")
    return passed


print("KFP components defined.")

In [ ]:
# ---------- Define the pipeline ----------
@dsl.pipeline(
    name="sklearn-training-pipeline",
    description="A simple pipeline: generate data -> train -> evaluate",
)
def training_pipeline(
    samples: int = 1000,
    n_estimators: int = 50,
    learning_rate: float = 0.1,
    accuracy_threshold: float = 0.85,
):
    # Step 1: Generate data
    data_task = generate_data(samples=samples)

    # Step 2: Train model
    train_task = train_sklearn_model(
        training_data=data_task.outputs["output_data"],
        n_estimators=n_estimators,
        learning_rate=learning_rate,
    )

    # Step 3: Evaluate
    eval_task = evaluate_model(
        model_artifact=train_task.outputs["model_artifact"],
        test_data=data_task.outputs["output_data"],
        threshold=accuracy_threshold,
    )


print("Pipeline defined.")

In [ ]:
# ---------- Compile the pipeline to JSON ----------
from kfp import compiler

compiler.Compiler().compile(
    pipeline_func=training_pipeline,
    package_path="sklearn_pipeline.json",
)
print("Pipeline compiled to sklearn_pipeline.json")

# Inspect the compiled spec
import json
with open("sklearn_pipeline.json") as f:
    spec = json.load(f)
print(f"Pipeline name: {spec.get('pipelineSpec', {}).get('pipelineInfo', {}).get('name', 'N/A')}")
print(f"Number of tasks: {len(spec.get('pipelineSpec', {}).get('root', {}).get('dag', {}).get('tasks', {}))}")

---
## 2. TFX Pipeline: Minimal ExampleGen + Trainer

TFX pipelines use standard components. Here we build the simplest possible TFX pipeline:
- **CsvExampleGen** — ingests CSV data and converts to tf.Example
- **Trainer** — trains a Keras model using the ingested data

In [ ]:
import os
import tempfile

# Create a small CSV dataset for the TFX pipeline
DATA_DIR = os.path.join(tempfile.gettempdir(), "tfx_data")
os.makedirs(DATA_DIR, exist_ok=True)

import csv
import random

random.seed(42)
csv_path = os.path.join(DATA_DIR, "data.csv")
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["feature_1", "feature_2", "feature_3", "label"])
    for _ in range(500):
        f1 = round(random.gauss(0, 1), 4)
        f2 = round(random.gauss(0, 1), 4)
        f3 = round(random.gauss(0, 1), 4)
        label = int(f1 + f2 > 0)
        writer.writerow([f1, f2, f3, label])

print(f"CSV created at {csv_path} with 500 rows.")

In [ ]:
# Write a TFX trainer module file
TRAINER_MODULE = os.path.join(tempfile.gettempdir(), "tfx_trainer_module.py")

trainer_code = '''
import tensorflow as tf
from tfx.components.trainer.fn_args_utils import FnArgs


FEATURE_KEYS = ["feature_1", "feature_2", "feature_3"]
LABEL_KEY = "label"


def _input_fn(file_pattern, batch_size=32):
    """Generate features and labels for training/eval."""
    feature_spec = {
        key: tf.io.FixedLenFeature([], tf.float32) for key in FEATURE_KEYS
    }
    feature_spec[LABEL_KEY] = tf.io.FixedLenFeature([], tf.int64)

    dataset = tf.data.TFRecordDataset(
        tf.io.gfile.glob(file_pattern), compression_type="GZIP"
    )

    def parse(serialized):
        parsed = tf.io.parse_single_example(serialized, feature_spec)
        label = parsed.pop(LABEL_KEY)
        features = tf.stack([parsed[k] for k in FEATURE_KEYS], axis=-1)
        return features, label

    return dataset.map(parse).batch(batch_size)


def run_fn(fn_args: FnArgs):
    """Train a simple Keras model."""
    train_dataset = _input_fn(fn_args.train_files[0] + "*")
    eval_dataset = _input_fn(fn_args.eval_files[0] + "*")

    model = tf.keras.Sequential([
        tf.keras.layers.Dense(16, activation="relu", input_shape=(3,)),
        tf.keras.layers.Dense(8, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    model.fit(train_dataset, validation_data=eval_dataset, epochs=5)
    model.save(fn_args.serving_model_dir)
'''

with open(TRAINER_MODULE, "w") as f:
    f.write(trainer_code)

print(f"Trainer module written to {TRAINER_MODULE}")

In [ ]:
# Build and run a minimal TFX pipeline locally
from tfx.components import CsvExampleGen, Trainer
from tfx.proto import trainer_pb2
from tfx.orchestration import pipeline as tfx_pipeline
from tfx.orchestration.local.local_dag_runner import LocalDagRunner

PIPELINE_NAME = "minimal-tfx-pipeline"
PIPELINE_ROOT = os.path.join(tempfile.gettempdir(), "tfx_pipeline_root")
METADATA_PATH = os.path.join(tempfile.gettempdir(), "tfx_metadata", "metadata.db")

# Components
example_gen = CsvExampleGen(input_base=DATA_DIR)

trainer = Trainer(
    module_file=TRAINER_MODULE,
    examples=example_gen.outputs["examples"],
    train_args=trainer_pb2.TrainArgs(num_steps=50),
    eval_args=trainer_pb2.EvalArgs(num_steps=10),
)

# Pipeline
tfx_local_pipeline = tfx_pipeline.Pipeline(
    pipeline_name=PIPELINE_NAME,
    pipeline_root=PIPELINE_ROOT,
    components=[example_gen, trainer],
    metadata_connection_config=tfx_pipeline.Pipeline.get_default_pipeline_config(
        pipeline_name=PIPELINE_NAME,
    ).metadata_connection_config if False else None,  # simplified for demo
)

print(f"TFX pipeline '{PIPELINE_NAME}' defined with {len(tfx_local_pipeline.components)} components.")
print("Components:", [c.id for c in tfx_local_pipeline.components])

# To actually run locally (may take a minute):
# LocalDagRunner().run(tfx_local_pipeline)
print("\n(Uncomment LocalDagRunner().run(...) above to execute the pipeline locally.)")

---
## 3. Vertex AI Pipeline: Compile and Submit

Submitting a KFP pipeline to Vertex AI Pipelines requires:
1. A compiled pipeline JSON (we already have this from Section 1)
2. A GCP project with Vertex AI API enabled
3. A service account with appropriate permissions

> **Cost Warning**: Submitting to Vertex AI will incur charges. Code is commented by default.

In [ ]:
# ============================================================
# VERTEX AI PIPELINE SUBMISSION (commented to avoid costs)
# ============================================================

# from google.cloud import aiplatform
#
# PROJECT_ID = "your-gcp-project-id"
# REGION = "us-central1"
# PIPELINE_ROOT_GCS = "gs://your-bucket/pipeline-root"
# SERVICE_ACCOUNT = "your-sa@your-project.iam.gserviceaccount.com"
#
# aiplatform.init(project=PROJECT_ID, location=REGION)
#
# # Submit the compiled pipeline
# job = aiplatform.PipelineJob(
#     display_name="sklearn-training-pipeline",
#     template_path="sklearn_pipeline.json",  # compiled in Section 1
#     pipeline_root=PIPELINE_ROOT_GCS,
#     parameter_values={
#         "samples": 2000,
#         "n_estimators": 100,
#         "learning_rate": 0.05,
#         "accuracy_threshold": 0.90,
#     },
#     enable_caching=True,  # skip unchanged steps on re-runs
# )
#
# job.submit(service_account=SERVICE_ACCOUNT)
# print(f"Pipeline submitted: {job.resource_name}")
#
# # Wait for completion (blocking)
# # job.wait()
# # print(f"Pipeline state: {job.state}")

print("Vertex AI submission code ready (commented to avoid costs).")
print("Uncomment and set PROJECT_ID, REGION, etc. to run on GCP.")

In [ ]:
# ============================================================
# SCHEDULING A PIPELINE ON VERTEX AI
# ============================================================

# from google.cloud.aiplatform import pipeline_jobs
#
# # Create a recurring schedule
# schedule = aiplatform.PipelineJobSchedule(
#     pipeline_job=job,
#     display_name="weekly-retrain-schedule",
# )
# schedule.create(
#     cron="0 2 * * 1",  # Every Monday at 2 AM
#     max_concurrent_run_count=1,
#     max_run_count=52,  # Run for 1 year
# )
# print(f"Schedule created: {schedule.resource_name}")

print("Pipeline scheduling example ready (commented).")

---
## 4. ML Metadata: Query Artifact Lineage

ML Metadata (MLMD) tracks every artifact, execution, and context in a pipeline run.
Here we demonstrate using the MLMD Python API with a local SQLite store.

In [ ]:
import ml_metadata as mlmd
from ml_metadata.metadata_store import metadata_store
from ml_metadata.proto import metadata_store_pb2

# Create an in-memory metadata store
connection_config = metadata_store_pb2.ConnectionConfig()
connection_config.sqlite.filename_uri = ":memory:"
store = metadata_store.MetadataStore(connection_config)

print("MLMD store created (in-memory SQLite).")

In [ ]:
# Register artifact types
dataset_type = metadata_store_pb2.ArtifactType(
    name="Dataset",
    properties={"split": metadata_store_pb2.STRING, "num_rows": metadata_store_pb2.INT}
)
dataset_type_id = store.put_artifact_type(dataset_type)

model_type = metadata_store_pb2.ArtifactType(
    name="Model",
    properties={"framework": metadata_store_pb2.STRING, "accuracy": metadata_store_pb2.DOUBLE}
)
model_type_id = store.put_artifact_type(model_type)

# Register execution type
train_type = metadata_store_pb2.ExecutionType(
    name="TrainModel",
    properties={"learning_rate": metadata_store_pb2.DOUBLE}
)
train_type_id = store.put_execution_type(train_type)

print(f"Registered types — Dataset: {dataset_type_id}, Model: {model_type_id}, TrainModel: {train_type_id}")

In [ ]:
# Create artifacts and an execution
# Input: training dataset
train_dataset = metadata_store_pb2.Artifact(
    type_id=dataset_type_id,
    uri="gs://my-bucket/data/train.csv",
    properties={
        "split": metadata_store_pb2.Value(string_value="train"),
        "num_rows": metadata_store_pb2.Value(int_value=10000),
    }
)
[train_dataset_id] = store.put_artifacts([train_dataset])

# Execution: training run
train_execution = metadata_store_pb2.Execution(
    type_id=train_type_id,
    properties={
        "learning_rate": metadata_store_pb2.Value(double_value=0.01),
    },
    last_known_state=metadata_store_pb2.Execution.COMPLETE,
)
[train_execution_id] = store.put_executions([train_execution])

# Output: trained model
trained_model = metadata_store_pb2.Artifact(
    type_id=model_type_id,
    uri="gs://my-bucket/models/v1/saved_model",
    properties={
        "framework": metadata_store_pb2.Value(string_value="sklearn"),
        "accuracy": metadata_store_pb2.Value(double_value=0.94),
    }
)
[trained_model_id] = store.put_artifacts([trained_model])

# Link: input event (dataset -> execution)
input_event = metadata_store_pb2.Event(
    artifact_id=train_dataset_id,
    execution_id=train_execution_id,
    type=metadata_store_pb2.Event.INPUT,
)

# Link: output event (execution -> model)
output_event = metadata_store_pb2.Event(
    artifact_id=trained_model_id,
    execution_id=train_execution_id,
    type=metadata_store_pb2.Event.OUTPUT,
)
store.put_events([input_event, output_event])

print(f"Created lineage: Dataset({train_dataset_id}) -> TrainModel({train_execution_id}) -> Model({trained_model_id})")

In [ ]:
# Query lineage: What inputs produced our model?
print("=" * 60)
print("LINEAGE QUERY: What produced the trained model?")
print("=" * 60)

# Get events for the model artifact
events = store.get_events_by_artifact_ids([trained_model_id])
for event in events:
    if event.type == metadata_store_pb2.Event.OUTPUT:
        # This model was an output of this execution
        exec_id = event.execution_id
        [execution] = store.get_executions_by_id([exec_id])
        exec_type = store.get_execution_types_by_id([execution.type_id])[0]
        print(f"\nProduced by execution: {exec_type.name} (id={exec_id})")
        print(f"  learning_rate = {execution.properties['learning_rate'].double_value}")

        # What were the inputs to that execution?
        input_events = store.get_events_by_execution_ids([exec_id])
        for ie in input_events:
            if ie.type == metadata_store_pb2.Event.INPUT:
                [input_artifact] = store.get_artifacts_by_id([ie.artifact_id])
                art_type = store.get_artifact_types_by_id([input_artifact.type_id])[0]
                print(f"\n  Input: {art_type.name} (id={ie.artifact_id})")
                print(f"    URI: {input_artifact.uri}")
                print(f"    split: {input_artifact.properties['split'].string_value}")
                print(f"    num_rows: {input_artifact.properties['num_rows'].int_value}")

---
## 5. Cloud Composer DAG Example

Cloud Composer runs Apache Airflow. Below is a complete DAG definition
that orchestrates an ML retraining workflow with BigQuery extraction
and Vertex AI custom training.

In [ ]:
# This is a Cloud Composer DAG definition (runs in Airflow, not in this notebook)
# Save this file to the DAGs folder in your Composer environment's GCS bucket

DAG_CODE = '''
from airflow import DAG
from airflow.operators.python import PythonOperator, BranchPythonOperator
from airflow.operators.dummy import DummyOperator
from airflow.providers.google.cloud.operators.bigquery import BigQueryInsertJobOperator
from airflow.providers.google.cloud.operators.vertex_ai.custom_job import (
    CreateCustomTrainingJobOperator,
)
from airflow.utils.trigger_rule import TriggerRule
from datetime import datetime, timedelta


PROJECT_ID = "my-gcp-project"
REGION = "us-central1"
DATASET = "ml_features"


default_args = {
    "owner": "ml-team",
    "retries": 2,
    "retry_delay": timedelta(minutes=5),
    "email_on_failure": True,
    "email": ["ml-team@company.com"],
}


def check_data_freshness(**kwargs):
    """Check if new data has arrived since last training run."""
    from google.cloud import bigquery
    client = bigquery.Client(project=PROJECT_ID)
    query = f"""
        SELECT COUNT(*) as new_rows
        FROM `{PROJECT_ID}.{DATASET}.transactions`
        WHERE DATE(created_at) >= DATE_SUB(CURRENT_DATE(), INTERVAL 7 DAY)
    """
    result = list(client.query(query).result())
    new_rows = result[0].new_rows
    if new_rows > 1000:
        return "extract_features"  # proceed with training
    return "skip_training"  # not enough new data


with DAG(
    dag_id="weekly_model_retrain",
    schedule_interval="0 3 * * 1",  # Every Monday at 3 AM
    start_date=datetime(2026, 1, 1),
    default_args=default_args,
    catchup=False,
    tags=["ml", "retraining"],
) as dag:

    # Branch: check if we have enough new data
    check_data = BranchPythonOperator(
        task_id="check_data_freshness",
        python_callable=check_data_freshness,
    )

    skip_training = DummyOperator(task_id="skip_training")

    # Extract features from BigQuery
    extract_features = BigQueryInsertJobOperator(
        task_id="extract_features",
        configuration={
            "query": {
                "query": f"SELECT * FROM `{PROJECT_ID}.{DATASET}.feature_table`",
                "destinationTable": {
                    "projectId": PROJECT_ID,
                    "datasetId": DATASET,
                    "tableId": "training_snapshot",
                },
                "writeDisposition": "WRITE_TRUNCATE",
            }
        },
    )

    # Train model on Vertex AI
    train = CreateCustomTrainingJobOperator(
        task_id="train_model",
        display_name="weekly-fraud-model",
        script_path="gs://my-bucket/trainer/task.py",
        container_uri="us-docker.pkg.dev/vertex-ai/training/tf-gpu.2-14:latest",
        model_serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/tf2-gpu.2-14:latest",
        region=REGION,
        project_id=PROJECT_ID,
    )

    # Notify on completion
    notify = PythonOperator(
        task_id="notify_team",
        python_callable=lambda: print("Training complete! Notify Slack."),
        trigger_rule=TriggerRule.NONE_FAILED,
    )

    # DAG structure
    check_data >> [extract_features, skip_training]
    extract_features >> train >> notify
    skip_training >> notify
'''

print("Cloud Composer DAG definition:")
print(DAG_CODE)

---
## 6. Pipeline Caching & Conditional Execution Patterns

### Caching
Vertex AI Pipelines caches component outputs by default. If a component's inputs, parameters,
and container image haven't changed, the cached output is reused.

### Conditional Execution
KFP v2 supports `dsl.If`, `dsl.Else`, and `dsl.OneOf` for conditional branching.

In [ ]:
# ---------- Conditional execution in KFP v2 ----------
from kfp import dsl


@dsl.component(base_image="python:3.10")
def check_accuracy(accuracy: float, threshold: float) -> bool:
    """Gate: returns True if accuracy meets threshold."""
    passed = accuracy >= threshold
    print(f"Accuracy {accuracy} vs threshold {threshold}: {'PASS' if passed else 'FAIL'}")
    return passed


@dsl.component(base_image="python:3.10")
def deploy_model(model_name: str) -> str:
    """Deploy the model to an endpoint."""
    print(f"Deploying model: {model_name}")
    return f"Deployed {model_name} successfully"


@dsl.component(base_image="python:3.10")
def send_alert(model_name: str, accuracy: float) -> str:
    """Send alert that model did not meet threshold."""
    msg = f"ALERT: {model_name} accuracy {accuracy} below threshold"
    print(msg)
    return msg


@dsl.pipeline(name="conditional-deploy-pipeline")
def conditional_pipeline(
    model_name: str = "fraud-detector-v2",
    accuracy: float = 0.92,
    threshold: float = 0.90,
):
    # Check if model passes quality gate
    check_task = check_accuracy(accuracy=accuracy, threshold=threshold)

    # Conditional: deploy if passed, alert if failed
    with dsl.If(check_task.output == True):  # noqa: E712
        deploy_task = deploy_model(model_name=model_name)

    with dsl.Else():
        alert_task = send_alert(model_name=model_name, accuracy=accuracy)


# Compile
from kfp import compiler

compiler.Compiler().compile(
    pipeline_func=conditional_pipeline,
    package_path="conditional_pipeline.json",
)
print("Conditional pipeline compiled to conditional_pipeline.json")

In [ ]:
# ---------- Caching control patterns ----------

# Pattern 1: Disable caching for a specific component
@dsl.pipeline(name="caching-demo")
def caching_pipeline(samples: int = 1000):
    # This step will be cached (default)
    data_task = generate_data(samples=samples)

    # This step has caching disabled — always re-runs
    train_task = train_sklearn_model(
        training_data=data_task.outputs["output_data"],
        n_estimators=100,
    )
    train_task.set_caching_options(False)  # <-- disable cache for this step


compiler.Compiler().compile(
    pipeline_func=caching_pipeline,
    package_path="caching_demo_pipeline.json",
)
print("Caching demo pipeline compiled.")

# Pattern 2: Force full re-execution at submission time
# job = aiplatform.PipelineJob(
#     ...,
#     enable_caching=False,  # disables caching for ALL components
# )

print("\nCaching patterns:")
print("  1. Per-component: task.set_caching_options(False)")
print("  2. Per-pipeline-run: enable_caching=False in PipelineJob")
print("  3. Default: caching enabled, skip steps with unchanged inputs")

---
## Summary

| Topic | Key Takeaway |
|-------|-------------|
| **KFP v2** | Framework-agnostic pipeline SDK; components are Python functions with typed I/O |
| **TFX** | TensorFlow-specific pipeline framework with built-in data validation and model analysis |
| **Vertex AI Pipelines** | Managed, serverless pipeline execution; runs KFP and TFX pipelines |
| **ML Metadata** | Tracks artifacts, executions, and lineage for auditability |
| **Cloud Composer** | Managed Airflow for cross-system orchestration (ETL + ML + notifications) |
| **Caching** | Enabled by default on Vertex AI; skips unchanged steps to save time and cost |
| **Conditional execution** | `dsl.If` / `dsl.Else` for branching logic in KFP v2 pipelines |